In [0]:
from pyspark.sql.functions import (
    avg,
    count,
    broadcast,
    col,
    when,
    least,
    lit,
    current_timestamp
)

checkpoint_base = (
    "/Volumes/fraud_detection/bronze/"
    "realtime_files/checkpoints"
)

# Historical behavior for each card
card_history = (
    spark.table("fraud_detection.silver.transactions")
    .groupBy("card_id")
    .agg(
        avg("amount").alias("previous_average_amount"),
        count("*").alias("historical_transaction_count")
    )
)

# Reference data
customers_ref = (
    spark.table("fraud_detection.silver.customers")
    .select(
        "customer_id",
        "risk_profile",
        "spending_profile",
        "credit_score"
    )
)

cards_ref = (
    spark.table("fraud_detection.silver.cards")
    .select(
        "card_id",
        col("customer_id").alias("card_customer_id"),
        "card_type",
        "status"
    )
)

merchants_ref = (
    spark.table("fraud_detection.silver.merchants")
    .select(
        "merchant_id",
        "merchant_category",
        "base_risk_score"
    )
)

# Read validated transactions
silver_stream = spark.readStream.table(
    "fraud_detection.silver.realtime_transactions"
)

# Stream-static enrichment
enriched_stream = (
    silver_stream
    .join(broadcast(customers_ref), "customer_id", "left")
    .join(broadcast(cards_ref), "card_id", "left")
    .join(broadcast(merchants_ref), "merchant_id", "left")
    .join(broadcast(card_history), "card_id", "left")
    .withColumn(
        "unusual_amount_flag",
        when(
            col("previous_average_amount").isNotNull() &
            (
                col("amount") >
                col("previous_average_amount") * 3
            ),
            1
        ).otherwise(0)
    )
)

In [0]:
scored_stream = (
    enriched_stream
    .withColumn(
        "amount_points",
        when(col("amount") >= 1000, 30)
        .when(col("amount") >= 500, 20)
        .otherwise(0)
    )
    .withColumn(
        "behavior_points",
        when(col("unusual_amount_flag") == 1, 25)
        .otherwise(0)
    )
    .withColumn(
        "customer_risk_points",
        when(col("risk_profile") == "high", 15)
        .when(col("risk_profile") == "medium", 7)
        .otherwise(0)
    )
    .withColumn(
        "merchant_risk_points",
        when(col("base_risk_score") >= 0.50, 20)
        .when(col("base_risk_score") >= 0.20, 10)
        .otherwise(0)
    )
    .withColumn(
        "credit_points",
        when(col("credit_score") < 550, 10)
        .otherwise(0)
    )
    .withColumn(
        "category_points",
        when(col("merchant_category") == "crypto_exchange", 10)
        .when(
            col("merchant_category").isin(
                "travel",
                "electronics"
            ),
            5
        )
        .otherwise(0)
    )
    .withColumn(
        "card_status_points",
        when(col("status") != "active", 30)
        .otherwise(0)
    )
    .withColumn(
        "risk_score",
        least(
            lit(100),
            col("amount_points") +
            col("behavior_points") +
            col("customer_risk_points") +
            col("merchant_risk_points") +
            col("credit_points") +
            col("category_points") +
            col("card_status_points")
        )
    )
    .withColumn(
        "risk_level",
        when(col("risk_score") >= 60, "high")
        .when(col("risk_score") >= 35, "medium")
        .otherwise("low")
    )
    .withColumn(
        "predicted_fraud",
        when(col("risk_score") >= 35, 1)
        .otherwise(0)
    )
    .withColumn(
        "recommended_action",
        when(
            col("risk_score") >= 60,
            "urgent_investigation"
        )
        .when(
            col("risk_score") >= 35,
            "manual_review"
        )
        .otherwise("approve")
    )
    .withColumn(
        "scored_timestamp",
        current_timestamp()
    )
)

In [0]:
score_query = (
    scored_stream.writeStream
    .format("delta")
    .option(
        "checkpointLocation",
        f"{checkpoint_base}/gold_realtime_scores"
    )
    .trigger(availableNow=True)
    .toTable(
        "fraud_detection.gold.realtime_scored_transactions"
    )
)

score_query.awaitTermination()

alerts_stream = (
    spark.readStream.table(
        "fraud_detection.gold.realtime_scored_transactions"
    )
    .filter(col("predicted_fraud") == 1)
    .select(
        "transaction_id",
        "customer_id",
        "card_id",
        "merchant_id",
        "amount",
        "transaction_timestamp",
        "previous_average_amount",
        "unusual_amount_flag",
        "risk_score",
        "risk_level",
        "recommended_action",
        "scored_timestamp"
    )
    .withColumn("alert_status", lit("open"))
    .withColumn(
        "alert_created_timestamp",
        current_timestamp()
    )
)

alert_query = (
    alerts_stream.writeStream
    .format("delta")
    .option(
        "checkpointLocation",
        f"{checkpoint_base}/gold_realtime_alerts"
    )
    .trigger(availableNow=True)
    .toTable(
        "fraud_detection.gold.realtime_fraud_alerts"
    )
)

alert_query.awaitTermination()

print(
    "Scored transactions:",
    spark.table(
        "fraud_detection.gold.realtime_scored_transactions"
    ).count()
)

print(
    "Fraud alerts:",
    spark.table(
        "fraud_detection.gold.realtime_fraud_alerts"
    ).count()
)